In [ ]:
from pathlib import Path
import re
import subprocess
import numpy as np
import pandas as pd
from scipy.optimize import root, minimize_scalar
from scipy.special import kv

c = 2.99792458e10
G = 6.67430e-8
h = 6.62607015e-27
kB = 1.380649e-16
sigma_sb = 5.670374419e-5
Msun = 1.98847e33
RYD_ERG = 13.605693122994 * 1.602176634e-12

ROOT = Path.cwd()
CLOUDY_EXE = Path('/home/maddaba/c25.00/source/cloudy.exe')
OUTPUT_DIR = ROOT / 'risultati_J152422_CLOUDY'
OBS_FILE = ROOT / 'Tab_completa_revisione.csv'
TRANS_FILE = ROOT / 'dati.asc'
TARGET = 'J152422.56'
EXCLUDE = {'Hd'}
SCALE_OBS = 1e42
FALLBACK_REL_ERR = 0.05

LOG_M = 9.0
CASES = [('SS', -1.8), ('ADAF', -2.2)]
LOG_N_H = 10.0   # n_H = 1e10 cm^-3, ipotesi sulla BLR        ################################################################# Ipotesi su BLR
LOG_RADIUS = 18.0  # R = 1e18 cm, ipotesi BLR                 ################################################################# Ipotesi su BLR
LOG_COLUMN = 23.0 # NH = 1e23 cm^-2, ipotesi BLR              ################################################################# Ipotesi su BLR

# Frequenze/righe di emissione: NON sono le soglie di ionizzazione in dati.asc.
# In caso di blend si sommano piu' transizioni solo quando il catalogo CLOUDY
# fornisce identificazioni univoche; non attribuiamo automaticamente i blend.
TARGET_WAVELENGTHS = {
    'Ha': (6562.8,), 'Halpha': (6562.8,),
    'Hb': (4861.3,), 'Hbeta': (4861.3,),
    'Hg': (4340.5,), 'Hgamma': (4340.5,),
    'Hd': (4101.7,), 'Hdelta': (4101.7,),
    'MgII': (2796.35, 2803.53),
    'CIII': (1908.73,), 'CIV': (1548.20, 1550.77),
}
# Filtro sulle etichette Cloudy; le righe fuori elenco sono lasciate non definite.
ION_HINTS = {
    'Ha': ('H  1','H 1','totl'), 'Halpha': ('H  1','H 1','totl'),
    'Hb': ('H  1','H 1','totl'), 'Hbeta': ('H  1','H 1','totl'),
    'Hg': ('H  1','H 1','totl'), 'Hgamma': ('H  1','H 1','totl'),
    'Hd': ('H  1','H 1','totl'), 'Hdelta': ('H  1','H 1','totl'),
    'MgII': ('Mg 2','Mg II'), 'CIII': ('C  3','C 3','C III'),
    'CIV': ('C  4','C 4','C IV'),
}

CFG = {
    'eta': 0.083, 'Nr': 80, 'Nnu_ss': 120,
    'nu_min_ss': 1e13, 'nu_max_ss': 1e17,
    'alfa': 0.3, 'rmin': 3.0, 'rmax': 1000.0,
    'delta': 0.0, 'beta_adaf': 0.5,
    'c1': 0.5, 'c3': 0.3, 'tnorm': 1.68e-10,
    'Nnu_adaf': 120, 'nu_min_adaf': 1e10, 'nu_max_adaf': 1e17,
    'root_y0': [0.5, 1.0, 0.99], 'root_tol': 1e-10,
    'logmdot_switch': -2.0,
}


def build_ss_spectrum(
    black_hole_mass_solar,
    eddington_accretion_rate,
    cfg
):
    """
    Compute the spectral energy distribution of a
    Shakura-Sunyaev thin accretion disk.

    Input:
        - black_hole_mass_solar:
          black-hole mass in solar masses

        - eddington_accretion_rate:
          accretion rate normalized to the Eddington value

        - cfg:
          configuration dictionary containing numerical
          and physical model parameters

    Processing:
        - compute Eddington luminosity
        - derive physical mass accretion rate
        - construct radial disk grid
        - compute disk temperature profile
        - evaluate blackbody emission at each radius
        - integrate emission over the entire disk

    Output:
        - frequency array
        - monochromatic luminosity spectrum (Lnu)
        - spectral energy distribution (nuLnu)
        - dictionary containing model information
    """

    # Convert black-hole mass from solar masses to CGS units
    black_hole_mass = black_hole_mass_solar * Msun

    # Retrieve model parameters from configuration
    radiative_efficiency = cfg["eta"]

    number_radial_points = cfg["Nr"]

    number_frequency_points = cfg["Nnu_ss"]

    minimum_frequency = cfg["nu_min_ss"]
    maximum_frequency = cfg["nu_max_ss"]

    # Compute Eddington luminosity
    eddington_luminosity = (
        1.3e38 * black_hole_mass_solar
    )

    # Compute total disk luminosity
    disk_luminosity = (
        eddington_accretion_rate
        * eddington_luminosity
    )

    # Convert luminosity into physical mass accretion rate
    mass_accretion_rate = (
        disk_luminosity
        / (radiative_efficiency * c**2)
    )

    # Compute characteristic disk radii
    schwarzschild_radius = (
        2 * G * black_hole_mass
        / c**2
    )

    inner_radius = 3 * schwarzschild_radius

    outer_radius = 3000 * schwarzschild_radius

    # Construct logarithmically spaced radial grid
    radius_grid = np.logspace(
        np.log10(inner_radius),
        np.log10(outer_radius),
        number_radial_points
    )

    # Inner-boundary correction factor
    boundary_factor = (
        1.0
        - np.sqrt(inner_radius / radius_grid)
    )

    boundary_factor = np.clip(
        boundary_factor,
        0.0,
        None
    )

    # Compute effective temperature profile
    temperature_profile = (
        (
            3
            * G
            * black_hole_mass
            * mass_accretion_rate
        )
        /
        (
            8
            * np.pi
            * sigma_sb
            * radius_grid**3
        )
        * boundary_factor
    )**0.25

    # Construct frequency grid
    frequency_grid = np.logspace(
        np.log10(minimum_frequency),
        np.log10(maximum_frequency),
        number_frequency_points
    )

    # Allocate Planck-function matrix
    planck_function = np.zeros(
        (
            number_frequency_points,
            number_radial_points
        )
    )

    # Evaluate blackbody emission at each radius
    for j in range(number_radial_points):

        if temperature_profile[j] <= 0:
            continue

        x = (
            h * frequency_grid
            /
            (
                kB
                * temperature_profile[j]
            )
        )

        # Prevent numerical overflow
        x = np.clip(
            x,
            1e-12,
            700.0
        )

        planck_function[:, j] = (
            (
                2
                * h
                * frequency_grid**3
                / c**2
            )
            /
            (
                np.exp(x) - 1.0
            )
        )

    # Integrate the contribution of all disk annuli
    integrand = (
        radius_grid[None, :]
        * planck_function
    )

    integrated_intensity = np.trapezoid(
        integrand,
        radius_grid,
        axis=1
    )

    # Convert intensity into luminosity
    luminosity_spectrum = (
        4                                        
        * np.pi**2
        * integrated_intensity
    )

    # Compute spectral energy distribution
    spectral_energy_distribution = (
        frequency_grid
        * luminosity_spectrum
    )

    # Store additional physical information
    model_info = {

        "regime": "SS",

        "black_hole_mass_solar":
            black_hole_mass_solar,

        "eddington_accretion_rate":
            eddington_accretion_rate,

        "eddington_luminosity":
            eddington_luminosity,

        "disk_luminosity":
            disk_luminosity,

        "mass_accretion_rate_cgs":
            mass_accretion_rate
    }

    return (
        frequency_grid,
        luminosity_spectrum,
        spectral_energy_distribution,
        model_info
    )

def bessel_k(order, argument):
    """
    Compute the modified Bessel function of the second kind.

    Input:
        - order: order of the Bessel function
        - argument: value at which the function is evaluated

    Output:
        - value of the modified Bessel function
    """

    return kv(order, argument)

def adaf_thermal_function(theta_e):
    """
    Compute the ADAF thermal correction function.

    Input:
        - theta_e: dimensionless electron temperature

    Processing:
        - evaluate the Bessel function required by the ADAF model
        - check that the result is finite and physically valid
        - compute the thermal correction term

    Output:
        - thermal correction factor
    """

    bessel_value = bessel_k(2, 1.0 / theta_e)

    if not np.isfinite(bessel_value) or bessel_value <= 0:
        return np.nan

    return (
        (1.0 / bessel_value)
        * (2.0 + 2.0 * theta_e + 1.0 / theta_e)
        * np.exp(-1.0 / theta_e)
    )

def adaf_temperature_equations(
    variables,
    accretion_rate,
    black_hole_mass,
    beta,
    cfg
):
    """
    Define the nonlinear ADAF equations solved by the root finder.

    Input:
        - variables: numerical variables explored by the solver
        - accretion_rate: Eddington-scaled accretion rate
        - black_hole_mass: black-hole mass in solar masses
        - beta: gas-to-total pressure parameter
        - cfg: configuration dictionary

    Processing:
        - transform solver variables into physical quantities
        - compute heating and cooling terms
        - compute synchrotron, Compton, and bremsstrahlung contributions
        - return the residuals of the ADAF balance equations

    Output:
        - list of three residuals to be driven close to zero
    """

    # Reject invalid numerical inputs
    if not np.all(np.isfinite(variables)):
        return [1e99, 1e99, 1e99]

    # Reject non-physical model parameters
    if (
        accretion_rate <= 0
        or black_hole_mass <= 0
        or beta <= 0
        or beta >= 1
    ):
        return [1e99, 1e99, 1e99]

    # Avoid extreme solver excursions
    if np.any(np.abs(variables) > 1e3):
        return [1e99, 1e99, 1e99]

    alpha = cfg["alfa"]
    inner_radius = cfg["rmin"]
    outer_radius = cfg["rmax"]
    electron_heating_fraction = cfg["delta"]
    c1 = cfg["c1"]
    c3 = cfg["c3"]
    temperature_normalization = cfg["tnorm"]

    # Squared variables guarantee positive physical quantities
    advection_fraction = variables[2]**2
    synchrotron_parameter = 1e3 * variables[1]**2
    theta_e = variables[0]**2

    if (
        advection_fraction <= 0
        or synchrotron_parameter <= 0
        or theta_e <= 1e-12
    ):
        return [1e99, 1e99, 1e99]

    thermal_factor = adaf_thermal_function(theta_e)

    if not np.isfinite(thermal_factor):
        return [1e99, 1e99, 1e99]

    # ADAF scaling coefficients
    synchrotron_scale_1 = (
        1.42e9
        * alpha**(-0.5)
        * np.sqrt(1 - beta)
        * c1**(-0.5)
        * c3**0.5
    )

    synchrotron_scale_2 = (
        1.19e-13
        * synchrotron_parameter
    )

    # Ion heating term
    ion_heating = (
        1.2e4
        * thermal_factor
        * alpha**(-2)
        * c1**(-2)
        * c3
        * beta
        * black_hole_mass
        * accretion_rate**2
        / inner_radius
    )

    # Direct electron heating term
    electron_heating = (
        electron_heating_fraction
        * 9.39e4
        * ((1 - beta) / advection_fraction)
        * c3
        * black_hole_mass
        * accretion_rate
        / inner_radius
    )

    total_heating = ion_heating + electron_heating

    peak_synchrotron_frequency = (
        synchrotron_scale_1
        * synchrotron_scale_2
        * black_hole_mass**(-0.5)
        * accretion_rate**0.5
        * (theta_e / temperature_normalization)**2
        * inner_radius**(-5/4)
    )

    if (
        not np.isfinite(peak_synchrotron_frequency)
        or peak_synchrotron_frequency <= 0
    ):
        return [1e99, 1e99, 1e99]

    synchrotron_power = (
        53.0
        * (synchrotron_parameter / 1000.0)**3
        * (theta_e / (1e9 * temperature_normalization))**7
        * black_hole_mass**0.5
        * accretion_rate**1.5
    )

    compton_amplification = (
        1
        + 4 * theta_e
        + 16 * theta_e**2
    )

    optical_depth = (
        23.87
        * accretion_rate
    )

    if optical_depth <= 0 or compton_amplification <= 1:
        return [1e99, 1e99, 1e99]

    compton_slope = (
        -np.log(optical_depth)
        / np.log(compton_amplification)
    )

    try:
        compton_base = (
            6.2e7
            * (theta_e / (1e9 * temperature_normalization))
            / (peak_synchrotron_frequency * 1e-12)
        )

        if (
            compton_base <= 0
            or not np.isfinite(compton_base)
        ):
            return [1e99, 1e99, 1e99]

        compton_factor = (
            compton_base**(1 - compton_slope)
            - 1
        )

        compton_power = (
            (synchrotron_power / 0.71)
            / (1 - compton_slope)
            * compton_factor
        )

    except Exception:
        return [1e99, 1e99, 1e99]

    # Bremsstrahlung correction factor
    if theta_e < 1:
        bremsstrahlung_factor = (
            4
            * np.sqrt(2 * theta_e / np.pi**3)
            * (1 + 1.781 * theta_e**1.34)
        )

    else:
        bremsstrahlung_factor = (
            (9 * theta_e / (2 * np.pi))
            * (
                np.log(1.123 * theta_e + 0.48)
                + 1.5
            )
        )

    bremsstrahlung_power = (
        4.78
        * alpha**(-2)
        * c1**(-2)
        * np.log(outer_radius / inner_radius)
        * bremsstrahlung_factor
        * black_hole_mass
        * accretion_rate**2
    )

    # First residual: heating minus total cooling
    residual_energy = (
        total_heating
        - synchrotron_power
        - compton_power
        - bremsstrahlung_power
    )

    # Second residual: advected energy balance
    residual_advection = (
        (1 - advection_fraction)
        * 9.39e4
        * ((1 - beta) / advection_fraction)
        * c3
        * black_hole_mass
        * accretion_rate
        / inner_radius
        - total_heating
    )

    bessel_value = bessel_k(2, 1 / theta_e)
    logarithm_argument = theta_e**3 * bessel_value

    if (
        not np.isfinite(logarithm_argument)
        or logarithm_argument <= 0
    ):
        return [1e99, 1e99, 1e99]

    transformed_synchrotron_parameter = (
        synchrotron_parameter**(1/3)
    )

    # Third residual: synchrotron self-absorption condition
    residual_synchrotron = (
        transformed_synchrotron_parameter
        + 1.852 * np.log(transformed_synchrotron_parameter)
        - 10.36
        - 0.26 * np.log(black_hole_mass * accretion_rate)
        + 0.26 * np.log(logarithm_argument)
    )

    residuals = np.array(
        [
            residual_energy,
            residual_synchrotron,
            residual_advection
        ],
        dtype=float
    )

    if not np.all(np.isfinite(residuals)):
        return [1e99, 1e99, 1e99]

    return residuals.tolist()

def build_adaf_spectrum_from_solution(
    electron_temperature,
    advection_fraction,
    synchrotron_parameter,
    black_hole_mass,
    accretion_rate,
    beta,
    cfg
):
    """
    Compute the ADAF spectrum after the physical solution is found.

    Input:
        - electron_temperature: electron temperature parameter
        - advection_fraction: fraction of energy advected inward
        - synchrotron_parameter: synchrotron self-absorption parameter
        - black_hole_mass: black-hole mass in solar masses
        - accretion_rate: Eddington-scaled accretion rate
        - beta: gas-to-total pressure parameter
        - cfg: configuration dictionary

    Processing:
        - build the frequency grid
        - compute synchrotron emission
        - compute Comptonized synchrotron emission
        - compute bremsstrahlung emission
        - sum all components

    Output:
        - logarithmic frequency grid
        - logarithmic spectral energy distribution
    """

    alpha = cfg["alfa"]
    inner_radius = cfg["rmin"]
    outer_radius = cfg["rmax"]
    c1 = cfg["c1"]
    c3 = cfg["c3"]
    temperature_normalization = cfg["tnorm"]

    minimum_frequency = cfg["nu_min_adaf"]
    maximum_frequency = cfg["nu_max_adaf"]
    number_frequency_points = cfg["Nnu_adaf"]

    log_frequency_grid = np.linspace(
        np.log10(minimum_frequency),
        np.log10(maximum_frequency),
        number_frequency_points
    )

    frequency_grid = 10**log_frequency_grid

    theta_e = (
        temperature_normalization
        * electron_temperature
    )

    if electron_temperature <= 0 or theta_e <= 0:
        raise RuntimeError("Invalid ADAF temperature")

    synchrotron_scale_1 = (
        1.42e9
        * alpha**(-0.5)
        * np.sqrt(1 - beta)
        * c1**(-0.5)
        * c3**0.5
    )

    synchrotron_scale_2 = (
        1.19e-13
        * synchrotron_parameter
    )

    synchrotron_scale_3 = 1.05e-24

    compton_amplification = (
        1
        + 4 * theta_e
        + 16 * theta_e**2
    )

    optical_depth = (
        23.87
        * accretion_rate
    )

    if optical_depth <= 0 or compton_amplification <= 1:
        raise RuntimeError("Invalid ADAF Compton parameters")

    compton_slope = (
        -np.log(optical_depth)
        / np.log(compton_amplification)
    )

    log_peak_synchrotron_frequency = np.log10(
        synchrotron_scale_1
        * synchrotron_scale_2
        * black_hole_mass**(-0.5)
        * accretion_rate**0.5
        * electron_temperature**2
        * inner_radius**(-5/4)
    )

    log_maximum_compton_frequency = (
        np.log10(electron_temperature)
        + 10.796
    )

    log_synchrotron_peak_luminosity = (
        3 * np.log10(synchrotron_scale_1 * synchrotron_scale_2)
        + np.log10(synchrotron_scale_3)
        - (7/4) * np.log10(inner_radius)
        + np.log10(black_hole_mass**0.5 * accretion_rate**1.5)
        + 7 * np.log10(electron_temperature)
    )

    log_synchrotron_luminosity = np.zeros(
        number_frequency_points
    )

    log_compton_luminosity = np.zeros(
        number_frequency_points
    )

    log_bremsstrahlung_luminosity = np.zeros(
        number_frequency_points
    )

    if theta_e < 1:
        bremsstrahlung_factor = (
            4
            * np.sqrt(2 * theta_e / np.pi**3)
            * (1 + 1.781 * theta_e**1.34)
        )

    else:
        bremsstrahlung_factor = (
            (9 * theta_e / (2 * np.pi))
            * (
                np.log(1.123 * theta_e + 0.48)
                + 1.5
            )
        )

    for i in range(number_frequency_points):

        # Synchrotron component below the peak frequency
        if log_frequency_grid[i] < log_peak_synchrotron_frequency:

            log_synchrotron_luminosity[i] = (
                np.log10(synchrotron_scale_3)
                + (8/5) * np.log10(
                    synchrotron_scale_1
                    * synchrotron_scale_2
                )
                + np.log10(
                    black_hole_mass**(6/5)
                    * accretion_rate**(4/5)
                )
                + (21/5) * np.log10(electron_temperature)
                + (2/5) * log_frequency_grid[i]
            )

        # Comptonized synchrotron component above the peak
        if log_frequency_grid[i] > log_peak_synchrotron_frequency:

            compton_base = (
                log_synchrotron_peak_luminosity
                + (compton_slope - 1)
                * log_peak_synchrotron_frequency
                - compton_slope
                * log_frequency_grid[i]
            )

            compton_cutoff = (
                -(
                    10**(
                        log_frequency_grid[i]
                        - log_maximum_compton_frequency
                    )
                )
                / np.log(10)
            )

            log_compton_luminosity[i] = (
                compton_base
                + compton_cutoff
            )

        bremsstrahlung_base = (
            24.36
            - 2 * np.log10(alpha * c1)
            + np.log10(
                np.log(outer_radius / inner_radius)
                * bremsstrahlung_factor
            )
            - np.log10(electron_temperature)
            + np.log10(
                black_hole_mass
                * accretion_rate**2
            )
        )

        bremsstrahlung_cutoff = (
            -(
                4.8e-11
                * frequency_grid[i]
                / electron_temperature
            )
            / np.log(10)
        )

        log_bremsstrahlung_luminosity[i] = (
            bremsstrahlung_base
            + bremsstrahlung_cutoff
        )

    # Total ADAF emission from all components
    total_flux = (
        10**log_synchrotron_luminosity
        + 10**log_compton_luminosity
        + 10**log_bremsstrahlung_luminosity
    )

    if (
        not np.all(np.isfinite(total_flux))
        or np.any(total_flux < 0)
    ):
        raise RuntimeError("Invalid ADAF flux")

    log_spectral_energy_distribution = (
        log_frequency_grid
        + np.log10(total_flux)
    )

    return (
        log_frequency_grid,
        log_spectral_energy_distribution
    )

def compute_adaf_spectral_data(
    black_hole_mass,
    accretion_rate,
    beta,
    cfg,
    verbose=False
):
    """
    Solve the ADAF equations and build the corresponding spectrum.

    Input:
        - black_hole_mass: black-hole mass in solar masses
        - accretion_rate: Eddington-scaled accretion rate
        - beta: gas-to-total pressure parameter
        - cfg: configuration dictionary
        - verbose: if True, print the current model parameters

    Processing:
        - initialize the root solver
        - solve the ADAF nonlinear equations
        - recover the physical ADAF quantities
        - compute the spectral energy distribution
        - convert logarithmic quantities into physical luminosities

    Output:
        - frequency grid
        - monochromatic luminosity spectrum
        - spectral energy distribution
        - model information dictionary
    """

    if verbose:
        print(
            f"ADAF -> m={black_hole_mass:.3e}, "
            f"mdot={accretion_rate:.3e}, beta={beta}"
        )

    initial_solution = cfg["root_y0"]

    solution = root(
        adaf_temperature_equations,
        initial_solution,
        args=(
            accretion_rate,
            black_hole_mass,
            beta,
            cfg
        ),
        tol=cfg["root_tol"]
    )

    if not solution.success:
        raise RuntimeError(
            f"Root did not converge for "
            f"m={black_hole_mass:.3e}, "
            f"mdot={accretion_rate:.3e}"
        )

    solver_variables = solution.x

    theta_e = solver_variables[0]**2

    electron_temperature = (
        theta_e
        / cfg["tnorm"]
    )

    synchrotron_parameter = (
        1e3
        * solver_variables[1]**2
    )

    advection_fraction = (
        solver_variables[2]**2
    )

    (
        log_frequency_grid,
        log_spectral_energy_distribution
    ) = build_adaf_spectrum_from_solution(
        electron_temperature,
        advection_fraction,
        synchrotron_parameter,
        black_hole_mass,
        accretion_rate,
        beta,
        cfg
    )

    frequency_grid = 10**log_frequency_grid

    log_luminosity_spectrum = (
        log_spectral_energy_distribution
        - log_frequency_grid
    )

    luminosity_spectrum = (
        10**log_luminosity_spectrum          
    )

    spectral_energy_distribution = (
        frequency_grid
        * luminosity_spectrum
    )

    model_info = {

        "regime": "ADAF",

        "black_hole_mass":
            black_hole_mass,

        "accretion_rate":
            accretion_rate,

        "beta":
            beta,

        "electron_temperature":
            electron_temperature,

        "synchrotron_parameter":
            synchrotron_parameter,

        "advection_fraction":
            advection_fraction,

        "root_success":
            solution.success
    }

    return (
        frequency_grid,
        luminosity_spectrum,
        spectral_energy_distribution,
        model_info
    )

def build_adaf_spectrum(
    black_hole_mass_solar,
    eddington_accretion_rate,
    cfg
):
    """
    High-level wrapper for the ADAF spectrum.

    Input:
        - black_hole_mass_solar: black-hole mass in solar masses
        - eddington_accretion_rate: Eddington-scaled accretion rate
        - cfg: configuration dictionary

    Processing:
        - read the ADAF beta parameter from the configuration
        - solve the ADAF physical equations
        - remove invalid numerical values
        - sort the spectrum by increasing frequency

    Output:
        - cleaned frequency grid
        - cleaned luminosity spectrum
        - cleaned spectral energy distribution
        - ADAF model information
    """

    beta = cfg["beta_adaf"]

    (
        frequency_grid,
        luminosity_spectrum,
        spectral_energy_distribution,
        model_info
    ) = compute_adaf_spectral_data(
        black_hole_mass=black_hole_mass_solar,
        accretion_rate=eddington_accretion_rate,
        beta=beta,
        cfg=cfg,
        verbose=False
    )

    # Keep only finite and physically meaningful values
    valid_mask = (
        np.isfinite(frequency_grid)
        & np.isfinite(luminosity_spectrum)
        & np.isfinite(spectral_energy_distribution)
        & (frequency_grid > 0)
        & (luminosity_spectrum >= 0)
    )

    frequency_grid = frequency_grid[valid_mask]
    luminosity_spectrum = luminosity_spectrum[valid_mask]
    spectral_energy_distribution = (
        spectral_energy_distribution[valid_mask]
    )

    if len(frequency_grid) < 2:
        raise RuntimeError(
            "Invalid ADAF spectrum after filtering"
        )

    sorting_index = np.argsort(frequency_grid)

    return (
        frequency_grid[sorting_index],
        luminosity_spectrum[sorting_index],
        spectral_energy_distribution[sorting_index],
        model_info
    )

def build_unified_spectrum(
    log_black_hole_mass,
    log_accretion_rate,
    cfg
):
    """
    Build the continuum spectrum using either the ADAF or SS model.

    Input:
        - log_black_hole_mass: base-10 logarithm of black-hole mass
        - log_accretion_rate: base-10 logarithm of accretion rate
        - cfg: configuration dictionary

    Processing:
        - convert logarithmic parameters into physical values
        - compare the accretion rate with the regime threshold
        - use the ADAF spectrum at low accretion rate
        - use the SS spectrum at high accretion rate

    Output:
        - frequency grid
        - luminosity spectrum
        - spectral energy distribution
        - model information dictionary
    """

    black_hole_mass = 10**log_black_hole_mass
    accretion_rate = 10**log_accretion_rate

    # Select the accretion regime according to the accretion rate
    if log_accretion_rate < cfg["logmdot_switch"]:

        (
            frequency_grid,
            luminosity_spectrum,
            spectral_energy_distribution,
            model_info
        ) = build_adaf_spectrum(
            black_hole_mass_solar=black_hole_mass,
            eddington_accretion_rate=accretion_rate,
            cfg=cfg
        )

        regime = "ADAF"

    else:

        (
            frequency_grid,
            luminosity_spectrum,
            spectral_energy_distribution,
            model_info
        ) = build_ss_spectrum(
            black_hole_mass_solar=black_hole_mass,
            eddington_accretion_rate=accretion_rate,
            cfg=cfg
        )

        regime = "SS"

    model_info["regime"] = regime

    return (
        frequency_grid,
        luminosity_spectrum,
        spectral_energy_distribution,
        model_info
    )


def read_obs_and_transitions():   
    if not OBS_FILE.is_file():
        raise FileNotFoundError(f'Manca {OBS_FILE}')
    if not TRANS_FILE.is_file():
        raise FileNotFoundError(f'Manca {TRANS_FILE}')
    obs = pd.read_csv(OBS_FILE)
    labcol = next((s for s in ['label','file_name','name','nome','blazar'] if s in obs), None)
    if labcol is None:
        raise ValueError('CSV privo della colonna identificativa del blazar')
    row = obs.loc[obs[labcol].astype(str).str.strip().eq(TARGET)]
    if len(row) != 1:
        raise ValueError(f'{TARGET}: trovate {len(row)} righe, selezione non univoca')
    row = row.iloc[0]
    selected=[]
    for col in obs.columns:
        if not col.endswith('_norm'):
            continue
        line = col[:-5]
        if line in EXCLUDE:
            continue
        val=pd.to_numeric(row[col],errors='coerce')
        if not np.isfinite(val) or val <= 0:
            continue
        ername=line+'_norm_unc'
        er=pd.to_numeric(row[ername],errors='coerce') if ername in obs.columns else np.nan
        if not np.isfinite(er) or er<=0:
            er=FALLBACK_REL_ERR*val
        selected.append(dict(line_name=line,L_obs=val*SCALE_OBS,sigma_L=er*SCALE_OBS))
    observed=pd.DataFrame(selected)
    if observed.empty:
        raise ValueError('Nessuna luminosita osservata valida per '+TARGET)
    # Verifica le transizioni: valori del file NON sono le lunghezze d'onda della riga.
    transitions=pd.read_csv(TRANS_FILE,sep=r'\s+',header=None,comment='#',
                            names=['line_name','E_ion_eV','nu_ion_Hz'],engine='python')
    transitions['line_name']=transitions['line_name'].astype(str).str.strip()
    observed=observed.merge(transitions,on='line_name',how='left',validate='many_to_one')
    print('Righe osservate:\n', observed.to_string(index=False))
    return observed


def save_sed(nu, lnu, path):                            ########################################################################## Punto in cui entra in gioco CLOUDY: si prendono le L_nu dnu e le trasforma in L_teoriche per ciascuna riga
    good=np.isfinite(nu)&np.isfinite(lnu)&(nu>0)&(lnu>0)  # Rimuove i valori non validi
    nu=np.asarray(nu)[good]; lnu=np.asarray(lnu)[good]   
    ix=np.argsort(nu); nu=nu[ix]; lnu=lnu[ix]     # Ordina per frequenza
    if len(nu)<3 or np.any(np.diff(nu)<=0):  
        raise ValueError('SED non valida')
    lint=np.trapezoid(lnu,nu)          # Luminosita integrata tra nu_min e nu_max, in erg/s
    if lint<=0 or not np.isfinite(lint):
        raise ValueError('Luminosita integrata della SED non valida')
    with path.open('w') as f:  # scrive il file di output per CLOUDY mettendo le frequenze in Rydberg e le luminosita normalizzate 
        f.write('# E(Ryd) e F_nu relativo; creati con il modello SS/ADAF originale\n')
        for v, l in zip(nu,lnu):
            f.write(f'{h*v/RYD_ERG:.12e} {l/lnu.max():.12e}\n')  # normalizza le luminosita alla massima luminosita della SED, non all'integrale!!!!
        f.write('***\n')
    return float(lint)


def run_case(label, logmdot):  # Funzione che esegue CLOUDY per un dato regime e logmdot, costruendo il file di input e salvando i risultati
    directory=OUTPUT_DIR/label
    directory.mkdir(parents=True,exist_ok=True)
    nu,lnu,sed,info=build_unified_spectrum(LOG_M,logmdot,CFG)
    if info['regime'] != label:
        raise RuntimeError(f'Regime non coerente: richiesto {label}, ottenuto {info["regime"]}')
    stem=f'{TARGET}_{label}'
    Lintegral=save_sed(nu,lnu,directory/(stem+'.sed'))
    lines=[
        f'title {TARGET} {label} logM={LOG_M} logmdot={logmdot}',
        f'table SED "{stem}.sed"',
        f'luminosity total {np.log10(Lintegral):.9f}',
        f'radius {LOG_RADIUS:.4f}',
        f'hden {LOG_N_H:.4f}',
        f'stop column density {LOG_COLUMN:.4f}',
        'iterate',
        # Sfera 4pi di riferimento (CF=1) per la normalizzazione dei flussi;
        # evitare il valore Cf=0.1 del MCMC originale.
        f'save last lines intensity "{stem}.int" emergent column',
        f'save last lines labels "{stem}.labels"',
        'database print',
        'print citation',
    ]
    (directory/(stem+'.in')).write_text('\n'.join(lines)+'\n')  #per sicurezza, terminare con newline
    run=subprocess.run([str(CLOUDY_EXE),'-r',stem],cwd=directory,  # esegue CLOUDY con il file di input appena creato, catturando l'output e gli errori
                       capture_output=True,text=True)
    out=directory/(stem+'.out')
    outtext=out.read_text(errors='replace') if out.exists() else ''   # legge il file di output, se esiste
    if run.returncode!=0 or 'Cloudy exited OK' not in outtext:
        raise RuntimeError('CLOUDY non terminato correttamente in '+str(directory)+'\n'+
                           run.stderr[-2000:]+'\n'+'\n'.join(outtext.splitlines()[-30:]))
    return directory,stem,info,Lintegral



# Lettura corretta delle intensita CLOUDY salvate con 'save lines intensity'
LINE_SPECS = {
    "MgII": [("Mg", 2, 2795.53), ("Mg", 2, 2802.71)],
    "CIII": [("C", 3, 1908.73)],
}

def parse_intensita_cloudy(path): # Funzione che legge il file .int generato da CLOUDY e restituisce un DataFrame con le intensita delle righe
    text = path.read_text(errors="replace")
    if "per unit inner area" not in text.splitlines()[0].lower():
        raise ValueError("Header di intensita CLOUDY inatteso: controllare le unita")
    # Per modelli iterati: consideriamo solo l'ultimo blocco di righe.
    if "begin emission lines" not in text.lower():
        raise ValueError("Sezione 'begin emission lines' assente")
    text = re.split(r"begin emission lines", text, flags=re.IGNORECASE)[-1]
    pat = re.compile(r"^\s*([A-Za-z]{1,2})\s+(\d+)\s+(\d+(?:\.\d+)?A)\s+([+\-]?[0-9.]+(?:[Ee][+\-]?\d+)?)\s*$")
    rows=[]
    for raw in text.splitlines():
        m=pat.match(raw)
        if m:
            rows.append(dict(element=m.group(1), stage=int(m.group(2)),
                             wavelength_A=float(m.group(3)[:-1]),
                             I_area=float(m.group(4))))
    if not rows:
        raise ValueError("Nessuna riga riconosciuta; ispezionare manualmente il file .int")
    return pd.DataFrame(rows)

def confronta_modello(regime, logmdot, force=False):   # Funzione che confronta le intensita osservate con quelle calcolate da CLOUDY per un dato regime e logmdot
    if not CLOUDY_EXE.is_file():
        raise FileNotFoundError(CLOUDY_EXE)
    obs = read_obs_and_transitions()
    directory = OUTPUT_DIR / regime
    stem = f"{TARGET}_{regime}"
    out_file = directory / (stem + ".out")
    int_file = directory / (stem + ".int")
    already_ok = (out_file.exists() and int_file.exists()
                  and "Cloudy exited OK" in out_file.read_text(errors="replace"))
    if already_ok and not force:
        print("Riutilizzo simulazione CLOUDY già completata:", out_file)
    else:
        print("Avvio CLOUDY:", regime, "logmdot =", logmdot)
        run_case(regime, logmdot)
    df = parse_intensita_cloudy(int_file)
    R = 10.0 ** LOG_RADIUS
    # Assunzione geometrica esplorativa: 4 pi R^2 per copertura completa.
    result=[]
    for _, row in obs.iterrows():
        name=row['line_name']
        components=LINE_SPECS.get(name)
        L=np.nan
        note='identificazione non implementata/verificata'
        if components:
            intensities=[]
            for elem, stage, lam in components:
                matches=df[(df.element==elem)&(df.stage==stage)&
                           ((df.wavelength_A-lam).abs()<0.10)]
                if len(matches)!=1:
                    intensities=[]
                    note=f"match {elem} {stage} {lam} A: {len(matches)} (serve verifica)"
                    break
                intensities.append(float(matches.iloc[0].I_area))
            if len(intensities)==len(components):
                L=4*np.pi*R**2*sum(intensities)
                note='L_4pi geometrica, da validare come luminosità emergente'
        result.append(dict(line_name=name,L_obs=row.L_obs,sigma_L=row.sigma_L,
                           L_CLOUDY_4pi=L, note=note))
    comparison=pd.DataFrame(result)
    comparison['rapporto_4pi_obs']=comparison.L_CLOUDY_4pi/comparison.L_obs
    comparison['regime']=regime
    comparison['logM']=LOG_M
    comparison['logmdot']=logmdot
    directory.mkdir(parents=True, exist_ok=True)
    path=directory/'confronto_intensita_4pi.csv'
    comparison.to_csv(path,index=False)
    print(comparison.to_string(index=False))
    print('CSV:',path)
    print('NOTA: queste sono luminosità equivalenti 4pi intrinseche, NON un fit del covering factor.')
    return comparison


In [ ]:
risultato_SS = confronta_modello("SS", -1.8)  

Righe osservate:
 line_name        L_obs      sigma_L  E_ion_eV    nu_ion_Hz
     MgII 1.579661e+41 4.873044e+39      7.60 1.838000e+15
     CIII 6.962898e+40 3.151367e+39     24.38 5.890000e+15
Avvio CLOUDY: SS logmdot = -1.8
line_name        L_obs      sigma_L  L_CLOUDY_4pi                                                    note  rapporto_4pi_obs regime  logM  logmdot
     MgII 1.579661e+41 4.873044e+39  1.938488e+43 L_4pi geometrica, da validare come luminosità emergente        122.715476     SS   9.0     -1.8
     CIII 6.962898e+40 3.151367e+39  2.063398e+40 L_4pi geometrica, da validare come luminosità emergente          0.296342     SS   9.0     -1.8
CSV: /home/maddaba/TESI/risultati_J152422_CLOUDY/SS/confronto_intensita_4pi.csv
NOTA: queste sono luminosità equivalenti 4pi intrinseche, NON un fit del covering factor.


In [15]:
risultato_ADAF = confronta_modello("ADAF", -2.2)

Righe osservate:
 line_name        L_obs      sigma_L  E_ion_eV    nu_ion_Hz
     MgII 1.579661e+41 4.873044e+39      7.60 1.838000e+15
     CIII 6.962898e+40 3.151367e+39     24.38 5.890000e+15
Avvio CLOUDY: ADAF logmdot = -2.2
line_name        L_obs      sigma_L  L_CLOUDY_4pi                                                    note  rapporto_4pi_obs regime  logM  logmdot
     MgII 1.579661e+41 4.873044e+39  3.397067e+41 L_4pi geometrica, da validare come luminosità emergente          2.150504   ADAF   9.0     -2.2
     CIII 6.962898e+40 3.151367e+39  2.971947e+38 L_4pi geometrica, da validare come luminosità emergente          0.004268   ADAF   9.0     -2.2
CSV: /home/maddaba/TESI/risultati_J152422_CLOUDY/ADAF/confronto_intensita_4pi.csv
NOTA: queste sono luminosità equivalenti 4pi intrinseche, NON un fit del covering factor.


In [16]:

from pathlib import Path
import re

base = (
    Path.cwd()
    / "risultati_J152422_CLOUDY"
    / "SS"
)

stem = "J152422.56_SS"

paths = {
    "input": base / f"{stem}.in",
    "output": base / f"{stem}.out",
    "intensita": base / f"{stem}.int",
    "etichette": base / f"{stem}.labels"
}

print("=" * 75)
print("VERIFICA OUTPUT CLOUDY C25 - MODELLO SS")
print("=" * 75)

print("\nFILE PRESENTI:")
for nome, path in paths.items():
    print(nome, ":", path.exists(), path)

for nome, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(path)

print("\n" + "=" * 75)
print("1. INPUT EFFETTIVAMENTE UTILIZZATO")
print("=" * 75)

print(paths["input"].read_text(errors="replace"))

print("\n" + "=" * 75)
print("2. INTESTAZIONE DEL FILE .int")
print("=" * 75)

with paths["intensita"].open(errors="replace") as f:
    for _ in range(5):
        print(repr(f.readline().rstrip("\n")))

print("\n" + "=" * 75)
print("3. SEZIONI DEL FILE .out")
print("=" * 75)

lines = paths["output"].read_text(
    errors="replace"
).splitlines()

for i, line in enumerate(lines):
    if re.search(
        r"intrinsic.*line|emergent.*line|"
        r"emission line spectrum",
        line,
        re.IGNORECASE
    ):
        print(f"Riga {i+1}: {line.strip()[:200]}")

print("\n" + "=" * 75)
print("4. RIGHE MgII E CIII] NEL .out")
print("=" * 75)

pattern = re.compile(
    r"(Mg\s+2\s+2795\.53A|"
    r"Mg\s+2\s+2802\.71A|"
    r"C\s+3\s+1908\.73A)",
    re.IGNORECASE
)

for i, line in enumerate(lines):
    if pattern.search(line):
        print(f"{i+1}: {line.strip()[:300]}")

print("\n" + "=" * 75)
print("5. ETICHETTE ATOMICHE NEL .labels")
print("=" * 75)

labels = paths["etichette"].read_text(
    errors="replace"
).splitlines()

for line in labels:
    if pattern.search(line):
        print(line[:250])

print("\n" + "=" * 75)
print("6. WARNING, CAUTION E PROBLEMI")
print("=" * 75)

for line in lines:
    if re.search(
        r"\b(DISASTER|PROBLEM|CAUTION|WARNING|"
        r"Cloudy exited OK)\b",
        line,
        re.IGNORECASE
    ):
        print(line.strip()[:250])

print("\n" + "=" * 75)
print("7. STATO FINALE DELLA SIMULAZIONE")
print("=" * 75)

print("\n".join(lines[-20:]))

print("\nDIAGNOSTICA COMPLETATA")


VERIFICA OUTPUT CLOUDY C25 - MODELLO SS

FILE PRESENTI:
input : True /home/maddaba/TESI/risultati_J152422_CLOUDY/SS/J152422.56_SS.in
output : True /home/maddaba/TESI/risultati_J152422_CLOUDY/SS/J152422.56_SS.out
intensita : True /home/maddaba/TESI/risultati_J152422_CLOUDY/SS/J152422.56_SS.int
etichette : True /home/maddaba/TESI/risultati_J152422_CLOUDY/SS/J152422.56_SS.labels

1. INPUT EFFETTIVAMENTE UTILIZZATO
title J152422.56 SS logM=9.0 logmdot=-1.8
table SED "J152422.56_SS.sed"
luminosity total 45.314854624
radius 18.0000
hden 10.0000
stop column density 23.0000
iterate
save last lines intensity "J152422.56_SS.int" emergent column
save last lines labels "J152422.56_SS.labels"
database print
print citation


2. INTESTAZIONE DEL FILE .int
'#Emission line intrinsic intensities per unit inner area'
'**********************************************************************************************************************************'
'title J152422.56 SS logM=9.0 logmdot=-1.8'
'table SED "J

In [17]:

from pathlib import Path

file_int = (
    Path.cwd()
    / "risultati_J152422_CLOUDY"
    / "SS"
    / "J152422.56_SS.int"
)

target = [
    "2795.53A",
    "2802.71A",
    "1908.73A",
    "1906.68A",
    "2798.00A",
    "1909.00A",
]

print("COLONNE ORIGINALI DEL FILE .int\n")

with file_int.open(errors="replace") as f:
    for line in f:
        if any(t in line for t in target):
            print("RIGA:", repr(line.rstrip()))
            print("COLONNE:", line.rstrip().split("\t"))
            print("-" * 70)


COLONNE ORIGINALI DEL FILE .int

RIGA: ' C  3                1908.73A \t1.642e+03'
COLONNE: [' C  3                1908.73A ', '1.642e+03']
----------------------------------------------------------------------
RIGA: ' C  3 Inwd           1908.73A \t0.000e+00'
COLONNE: [' C  3 Inwd           1908.73A ', '0.000e+00']
----------------------------------------------------------------------
RIGA: ' C  3 Coll           1908.73A \t0.000e+00'
COLONNE: [' C  3 Coll           1908.73A ', '0.000e+00']
----------------------------------------------------------------------
RIGA: ' C  3 Pump           1908.73A \t0.000e+00'
COLONNE: [' C  3 Pump           1908.73A ', '0.000e+00']
----------------------------------------------------------------------
RIGA: ' C  3                1906.68A \t1.460e-01'
COLONNE: [' C  3                1906.68A ', '1.460e-01']
----------------------------------------------------------------------
RIGA: ' C  3 Inwd           1906.68A \t0.000e+00'
COLONNE: [' C  3 Inwd      